# 07 Video Branch Debug

Use this notebook when you want to enable real video frames for the 3D CNN branch. It samples a small clip first so you can check speed and memory before full training.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import ARCHIVE_DIR, ALIGNED_DIR, FEATURES_DIR
from src.data.archive_scanner import scan_archive
from src.models.datasets import AlignedWindowDataset, add_window_targets, create_window_index
from src.models.dl_models import DLModelConfig, MultimodalPhysiologyNet


In [ ]:
records = scan_archive(ARCHIVE_DIR)
features = pd.read_csv(FEATURES_DIR / 'trial_features.csv')
windows = add_window_targets(create_window_index(ALIGNED_DIR, archive_records=records), features)
dataset = AlignedWindowDataset(windows.head(2), feature_table=features, sequence_length=64, video_frames=8, frame_size=64, use_video=True)
sample = dataset[0]
{key: tuple(value.shape) for key, value in sample.items() if torch.is_tensor(value)}


In [ ]:
batch = {key: value.unsqueeze(0) if torch.is_tensor(value) else value for key, value in sample.items()}
config = DLModelConfig(
    video_channels=batch['video'].shape[1],
    bvp_channels=batch['bvp'].shape[1],
    physiology_channels=batch['physiology'].shape[1],
    static_dim=batch['static'].shape[1],
)
model = MultimodalPhysiologyNet(config)
out = model(batch['video'], batch['bvp'], batch['physiology'], batch['static'])
{key: tuple(value.shape) for key, value in out.items()}
